# Learn representations with I-JEPA on Oxford Flowers

You will train an I-JEPA encoder on Oxford Flowers 102: a ViT that sees the visible patches of an image and predicts, in representation space, the embeddings of the patches that were hidden. No pixel reconstruction, no augmentation pipeline. After training you will measure the representations with a linear probe and a kNN probe, watch the collapse telemetry over the run, and pull the nearest neighbours of query images out of the frozen embeddings.

I-JEPA trains by prediction in latent space. A context encoder sees only some patches, a target encoder sees the whole image, and a narrow predictor maps the context embeddings plus the target positions to the target embeddings. The target encoder is not trained by gradients: it is an exponential moving average of the context encoder, so the targets keep improving as the encoder does, and there is no pixel-level shortcut to lower the loss.

**Expected time.** About 25 minutes on an A100, 30 to 40 on a TPU v5e, and 45 to 60 on an RTX 4080, for 100 epochs of 124 steps at 224 px with a ViT-S width encoder. A CPU run is not realistic at this configuration.

In [ ]:
# Install cell: only runs on Colab (import google.colab succeeds there).
# A TPU runtime gets jax[tpu] instead of jax[cuda12]. Locally this cell does nothing.
import os
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    jax_spec = "jax[tpu]" if "COLAB_TPU_ADDR" in os.environ else "jax[cuda12]"
    %pip install -q "dew-ml[tfds] @ git+https://github.com/AshishKumar4/dew" {jax_spec} tensorflow-datasets

In [ ]:
# Every size knob in one place.
IMAGE_SIZE = 224        # pixels; patch 16 gives the 14x14 grid of the paper
PATCH_SIZE = 16
BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 1e-3
EMB_FEATURES = 384      # encoder width (ViT-S)
NUM_LAYERS = 12
NUM_HEADS = 6
NUM_TARGET_BLOCKS = 4  # blocks the encoder must predict, hidden from it
PROBE_CLASSES = 102
VAL_RECORDS = 512       # held out from the head of the dataset, canonical order
OUT_DIR = "flowers-jepa"

In [ ]:
import jax
print("devices:", jax.devices())
print("backend:", jax.default_backend())

## The data

Oxford Flowers 102 is 8189 labelled flower photographs, downloaded from TensorFlow Datasets on first use. The grain loader resizes each record to 224 px and carries its class index through as `label`, which is what the probes will score against. The first `VAL_RECORDS` images in canonical order are the validation split, so the probes never score an image the encoder trained on.

The augmentation is deliberately nothing. In I-JEPA the variation comes from the masking, not from flips and jitters.

In [ ]:
from dew.data.dataloaders import get_dataset_grain

data = get_dataset_grain("oxford_flowers102", batch_size=BATCH_SIZE,
                         image_scale=IMAGE_SIZE, worker_count=2,
                         val_count=VAL_RECORDS, val_batch_size=256)
print("train:", data["train_len"], "val:", data["val_len"])

batch = next(iter(data["train"]()))
print(batch["image"].shape, batch["image"].dtype, "labels:", batch["label"][:8])

## The mask

Each image gets `NUM_TARGET_BLOCKS` rectangular blocks of patches to predict, drawn with a random scale and aspect ratio in the ranges the I-JEPA paper uses, and the context is a random subset of everything left. The geometry is resolved once for the 14x14 grid and each step samples block shapes and positions from it, so every mask has the same shape and the training step stays compiled.

The figure below is one sampled mask. The encoder sees the context patches and has to predict the embeddings of the target blocks, whose locations it is told but whose contents it is not.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from dew.objectives.jepa import multi_block_mask

GRID = (IMAGE_SIZE // PATCH_SIZE, IMAGE_SIZE // PATCH_SIZE)
mask = multi_block_mask(GRID, num_targets=NUM_TARGET_BLOCKS)
print("grid", GRID, "| context tokens:", mask.num_context,
      "| targets:", mask.num_targets, "blocks x", mask.block_area, "tokens each")

context_idx, target_idx = mask.sample(jax.random.PRNGKey(0), 1)
grid_view = np.zeros(GRID[0] * GRID[1])
grid_view[context_idx[0]] = 1
grid_view[target_idx.reshape(-1)] = 2
plt.figure(figsize=(3, 3))
plt.imshow(grid_view.reshape(GRID), cmap="coolwarm", interpolation="nearest")
plt.title("blue: context   red: targets")
plt.axis("off")
plt.show()

## Encoder, predictor, target encoder

The encoder is a ViT (`jepa_encoder`). The predictor is a narrower transformer (`jepa_predictor`) that reads the context embeddings plus mask tokens standing in for the targets, and outputs embeddings at the encoder's width. The target encoder has no parameters of its own: it is the EMA copy of the context encoder that the trainer already maintains, and the objective's `EMASpec` restricts that average to the `context_encoder` subtree so the predictor is left out of it.

The loss is the mean squared distance between predictions and layer-normalized targets, in fp32. The layer norm has no learned affine, which fixes the scale of the prediction problem: shrinking the embeddings is not a way to lower the loss. Two telemetry numbers ship with every step, because this objective fails silently. `repr_std` is how much the embeddings vary across a batch; it goes to zero exactly when the encoder stops distinguishing inputs. `repr_cov_offdiag` is the magnitude of the off-diagonal covariance; it rises when dimensions become redundant, which can happen while `repr_std` still looks healthy.

In [ ]:
import optax
from dew.inputs import DiffusionInputConfig
from dew.objectives.jepa import JepaObjective
from dew.objectives.jepa.probes import get_knn_probe_metric, get_linear_probe_metric
from dew.registry import apply_precision_policy, build_model

encoder_config = apply_precision_policy("jepa_encoder", dict(
    patch_size=PATCH_SIZE, emb_features=EMB_FEATURES,
    num_layers=NUM_LAYERS, num_heads=NUM_HEADS,
), dtype="bfloat16", attention_impl="auto")
encoder = build_model("jepa_encoder", encoder_config)
predictor = build_model("jepa_predictor", dict(
    grid=GRID, emb_features=EMB_FEATURES, predictor_features=EMB_FEATURES // 2,
    num_layers=NUM_LAYERS // 2, num_heads=NUM_HEADS,
    dtype=encoder_config["dtype"], attention_impl=encoder_config["attention_impl"]))

objective = JepaObjective(encoder, predictor, mask=mask,
                          sample_data_key="image",
                          sample_data_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))

## Training, with the telemetry recorded

The trainer is the same one the diffusion and language model notebooks use. The two probes run at the end of every epoch on the frozen EMA embeddings: a logistic regression fit on half of each validation batch and scored on the other half, and a cosine kNN classifier fit the same way. Chance is about 1 percent on 102 classes.

The collapse telemetry is collected through the same metric seam. Each of the two metrics below computes one of the health numbers on the embeddings validation produced and appends it to a list, so the notebook can plot it over the run afterwards. `higher_is_better` only affects how the trainer tracks the best value; both directions are recorded here for the curve, not for a leaderboard.

In [ ]:
from dew.eval.common import EvaluationMetric
from dew.objectives.jepa import representation_health
from dew.training import ObjectiveTrainer

std_history, cov_history = [], []

def record_std(embeddings, batch):
    value = float(representation_health(embeddings)["repr_std"])
    std_history.append(value)
    return value

def record_cov(embeddings, batch):
    value = float(representation_health(embeddings)["repr_cov_offdiag"])
    cov_history.append(value)
    return value

trainer = ObjectiveTrainer(
    encoder, optax.adamw(LEARNING_RATE), objective=objective,
    input_config=DiffusionInputConfig(sample_data_key="image",
                                       sample_data_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
                                       conditions=[]),
    eval_metrics=[
        get_linear_probe_metric(PROBE_CLASSES),
        get_knn_probe_metric(PROBE_CLASSES),
        EvaluationMetric(record_std, name="repr_std", higher_is_better=True),
        EvaluationMetric(record_cov, name="repr_cov_offdiag", higher_is_better=False),
    ],
    rngs=jax.random.PRNGKey(0), name=OUT_DIR,
    checkpoint_base_path="./checkpoints/flowers-jepa")
state = trainer.fit(data, training_steps_per_epoch=data["train_len"] // BATCH_SIZE,
                    epochs=EPOCHS, val_steps_per_epoch=2)

## What the curves say

The probes tell you whether the representations separate the classes without ever having seen a label. Anything in the tens of percent after 100 epochs on 8k images is a healthy encoder; the paper's numbers come from ImageNet-scale data and 2000 epochs, so do not compare them directly.

`repr_std` should start near 1 and stay well away from zero. `repr_cov_offdiag` should stay small; a steady climb while the probes stall is dimensional collapse, the other failure mode. The dashed lines mark the first epoch, before any learning.

In [ ]:
def per_epoch(history):
    return np.asarray(history).reshape(EPOCHS, -1).mean(axis=1)

epochs = np.arange(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(epochs, per_epoch(std_history))
axes[0].axhline(per_epoch(std_history)[0], linestyle="--", color="gray")
axes[0].set_title("repr_std (away from 0 is healthy)")
axes[1].plot(epochs, per_epoch(cov_history))
axes[1].axhline(per_epoch(cov_history)[0], linestyle="--", color="gray")
axes[1].set_title("repr_cov_offdiag (low is healthy)")
for ax in axes:
    ax.set_xlabel("epoch")
plt.tight_layout()
plt.show()

## Nearest neighbours

The point of the encoder is the embedding space, so look at it directly. Pool the frozen EMA encoder's tokens for each validation image, normalise the embeddings, and take cosine neighbours of a few queries. Each row is one query followed by its four nearest neighbours. Flowers of the same species clustering together is the same thing the probes measured numerically.

In [ ]:
# The validation step is the objective's: pooled embeddings from the frozen EMA encoder.
embed = objective.make_validation_step()
val_images, val_labels = [], []
for val_batch in iter(data["val"]()):
    val_images.append(np.asarray(val_batch["image"]))
    val_labels.append(np.asarray(val_batch["label"]))
val_images = np.concatenate(val_images)
val_labels = np.concatenate(val_labels)
embeddings = np.asarray(embed(state, {"image": val_images}))
embeddings = embeddings / (np.linalg.norm(embeddings, axis=-1, keepdims=True) + 1e-8)
print(val_images.shape, embeddings.shape)

queries = [0, 1, 2, 3]
fig, axes = plt.subplots(len(queries), 5, figsize=(12, 2.6 * len(queries)))
for row, q in enumerate(queries):
    similarity = embeddings @ embeddings[q]
    neighbours = np.argsort(-similarity)[1:5]
    for col, idx in enumerate([q, *neighbours]):
        ax = axes[row, col]
        ax.imshow(val_images[idx].astype(np.uint8))
        tag = "query" if col == 0 else f"nn {col}"
        ax.set_title(f"{tag} | label {val_labels[idx]}", fontsize=8)
        ax.axis("off")
plt.tight_layout()
plt.show()

## Keeping the encoder

The thing to keep from a JEPA run is the EMA of the context encoder, without the predictor. `save_params` writes it as a safetensors file under the names the tree uses, so anything that reads safetensors reads it back. The trainer's checkpoints under `./checkpoints/flowers-jepa` hold the full train state, including the optimizer, if you want to resume.

In [ ]:
from dew.interop import save_params

save_params(state.ema_params["params"]["context_encoder"], f"{OUT_DIR}/encoder.safetensors")
print("wrote", f"{OUT_DIR}/encoder.safetensors")

## Where to go next

The paper's ViT-H/16 trains 300 epochs on ImageNet with far more target blocks; the knobs are the ones this notebook set once (`NUM_TARGET_BLOCKS`, `EPOCHS`, the model width, the mask scale and aspect ranges in `multi_block_mask`). `recipes/jepa/train.py` runs the same configuration from the command line, and `jepa_video_encoder` with a `factorized=True` predictor does the same job on video clips.